# Extract DupCaller-unique mutation pileups

This notebook identifies mutations called exclusively by DupCaller (not by deepUMIcaller)
and extracts per-position pileup data from **four deepUMIcaller BAMs** and the **DupCaller markdup BAM**.


### BAMs extracted (basic pileup)

| Stage | BAM type | Pipeline step | Output folder |
|-------|----------|--------------|---------------|
| **0** | Raw aligned | `sortbamclean` | `0_raw_bam/` |
| **1** | Filtered consensus | `sortbamduplexfiltered` | `1_sortbamduplexfiltered/` |
| **2** | All-molecules consensus | `sortbamduplexclean` | `2_sortbamduplexclean/` |
| **3** | Duplex consensus | `sortbamduplexconsmed` | `3_sortbamduplexconsmed/` |

### BAMs extracted (duplicate-aware pileup)

| BAM type | Pipeline step | Output folder |
|----------|--------------|---------------|
| **DupCaller markdup** | DupCaller pipeline | `dupcaller_markdup_bam/` |

The duplicate-aware pileup splits read counts by the GATK `0x400` duplicate flag,
so we can distinguish ALT support from primary vs duplicate-flagged reads.
This is essential because DupCaller includes duplicate-flagged reads in its genotyping
but `extractDepthSnv` skips them (see `DUPCALLER_ALT0_ANALYSIS.md`).

All results are saved as TSV files under `results/dupcaller_unique/`.
The downstream notebook `diagnose_dupcaller_unique.ipynb` loads these TSVs directly — no BAM access needed.

In [1]:
# --- 1. Setup: imports --------------------------------------------------------

import re
import os
import contextlib
import pysam
import pandas as pd
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# --- 2. Analysis settings -----------------------------------------------------
cohort = "bladder"          # "cord_blood" or "bladder"
mutations = "snv"           # "snv" or "all_mutation_types"

# Base directory for all outputs
OUTPUT_BASE = Path("./results/dupcaller_unique")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

In [3]:
# --- 3. Cohort-specific paths -------------------------------------------------
if cohort == "bladder":
    deepcsa_path = "/data/bbg/nobackup2/scratch/mhuertas/tests_dupcaller/20260318_bladder_same/mutations"
    samples = [
        "P19_0050_BDO_01", "P19_0050_BTR_01",
        "P19_0051_BDO_01", "P19_0051_BTR_01",
        "P19_0052_BDO_01", "P19_0052_BTR_01",
        "P19_0053_BDO_01", "P19_0053_BTR_01",
    ]
    dupcaller_samples = [f"{sample}_dupcaller" for sample in samples]
    dupcaller_maf_path = "/data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/deepCSA_input/vcf/all_samples_info.maf"

# BAM base directories
DEEPUMI_BAM_BASE = "/data/bbg/nobackup2/prominent/bladder/2025-04-25_deepUMIcaller_new_samples"
DUPCALLER_BAM_BASE = "/data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035"

# Derived mutation table paths
all_both = f"{deepcsa_path}/germline_somatic/all_samples.filtered.tsv.gz"
all_deepumi = f"{deepcsa_path}/germline_somatic/CallerDeepumicaller.filtered.tsv.gz"
deepumi_clean = f"{deepcsa_path}/clean_somatic/CallerDeepumicaller.somatic.mutations.tsv"

# ── BAM path helpers ──────────────────────────────────────────────────────────
# deepUMIcaller pipeline BAMs (in processing order):

def get_raw_bam(sample):
    """Raw aligned reads, before UMI grouping (sortbamclean)."""
    # Changed the path because I had to sort it again and index it
    return f"/scratch/bbg/work/DupCaller/bam_analysis/sortbamclean/{sample}.sorted.sorted.bam"

def get_filtered_consensus_bam(sample):
    """Consensus reads after AS-XS filtering, before FilterConsensusReads/ClipBam (sortbamduplexfiltered)."""
    # Changed the path because I had to sort it again and index it
    return f"/scratch/bbg/work/DupCaller/bam_analysis/sortbamduplexfiltered/{sample}.sorted.sorted.bam"

def get_all_molecules_bam(sample):
    """All-molecules consensus BAM, after FilterConsensusReadsAM + ClipBamAM (sortbamduplexclean)."""
    return f"{DEEPUMI_BAM_BASE}/sortbamduplexclean/{sample}.sorted.bam"

def get_duplex_consensus_bam(sample):
    """Strict duplex consensus BAM, after FilterConsensusReadsDuplex + ClipBam (sortbamduplexconsmed)."""
    return f"{DEEPUMI_BAM_BASE}/sortbamduplexconsmed/{sample}.sorted.bam"

# DupCaller BAM:

def get_dupcaller_markdup_bam(sample):
    """Raw markdup BAM used by DupCaller."""
    return f"{DUPCALLER_BAM_BASE}/{sample}/markdup/{sample}.mkdped.bam"

print(f"Cohort: {cohort}  |  Mutation filter: {mutations}  |  Samples: {len(samples)}")

Cohort: bladder  |  Mutation filter: snv  |  Samples: 8


## 4. Identify DupCaller-unique mutations

Load caller outputs, merge, and extract mutations that:
- PASS in DupCaller
- Are NOT called (or not PASS) in deepUMIcaller
- Have F1R2 > 1 and F2R1 > 1 in DupCaller

In [4]:
# --- 4a. Helper functions -----------------------------------------------------

def add_variant_id(df: pd.DataFrame) -> pd.DataFrame:
    """Create a unique variant identifier as CHROM:POS_REF>ALT."""
    df["VARIANT_ID"] = (
        df["CHROM"].astype(str) + ":"
        + df["POS"].astype(str) + "_"
        + df["REF"].astype(str) + ">"
        + df["ALT"].astype(str)
    )
    return df


def extract_info_field(info_series: pd.Series, field: str) -> pd.Series:
    """Extract a numeric value for *field* from a VCF-style INFO string."""
    pattern = rf"(?:^|;){re.escape(field)}=([^;]+)"
    return info_series.str.extract(pattern, expand=False).astype(float)


def build_merged_variants(
    sample_deepumi_df: pd.DataFrame,
    sample_dupcaller_df: pd.DataFrame,
    deepumi_pass_only: bool = False,
    dupcaller_pass_only: bool = False,
) -> pd.DataFrame:
    """Outer-merge deepUMI and DupCaller variant tables and label each row."""
    if deepumi_pass_only:
        deepumi_subset = sample_deepumi_df[sample_deepumi_df["is_pass"] == True].copy()
    else:
        deepumi_subset = sample_deepumi_df.copy()

    if dupcaller_pass_only:
        dupcaller_subset = sample_dupcaller_df[sample_dupcaller_df["FILTER"] == "PASS"].copy()
    else:
        dupcaller_subset = sample_dupcaller_df.copy()

    deepumi_subset = deepumi_subset[
        ["VARIANT_ID", "SAMPLE_ID", "FILTER", "VAF", "TYPE", "DEPTH", "ALT_DEPTH"]
    ].rename(columns={
        "FILTER": "FILTER_deepumi", "VAF": "VAF_deepumi", "TYPE": "TYPE_deepumi",
        "DEPTH": "DEPTH_deepumi", "ALT_DEPTH": "ALT_DEPTH_deepumi",
    })

    dupcaller_subset = dupcaller_subset[
        ["SAMPLE_ID", "VARIANT_ID", "FILTER", "LR", "TYPE", "DEPTH", "ALT_DEPTH", "INFO"]
    ].rename(columns={
        "FILTER": "FILTER_dupcaller", "LR": "LR_dupcaller", "TYPE": "TYPE_dupcaller",
        "DEPTH": "DEPTH_dupcaller", "ALT_DEPTH": "ALT_DEPTH_dupcaller",
        "INFO": "INFO_dupcaller",
    })

    dupcaller_subset["SAMPLE_ID"] = dupcaller_subset["SAMPLE_ID"].str.replace(
        "_dupcaller", "", regex=False
    )

    merged_df = pd.merge(
        deepumi_subset, dupcaller_subset,
        on=["VARIANT_ID", "SAMPLE_ID"], how="outer", indicator=True,
    )
    status_map = {"both": "Common", "left_only": "Only deepUMI", "right_only": "Only DupCaller"}
    merged_df["status"] = merged_df["_merge"].map(status_map)
    return merged_df.drop(columns=["_merge"])


def parse_variant_id(variant_id: str) -> tuple[str, int, str, str]:
    """Parse 'chr:pos_REF>ALT' into (chrom, pos, ref, alt). pos is 1-based."""
    chrom, rest = variant_id.split(":")
    pos_str, alleles = rest.split("_")
    ref, alt = alleles.split(">")
    return chrom, int(pos_str), ref, alt

In [5]:
# --- 4b. Load and prepare caller data ----------------------------------------

dupcaller_df = pd.read_csv(dupcaller_maf_path, sep="\t", low_memory=False)
deepumi_df = pd.read_csv(all_deepumi, sep="\t", low_memory=False)
deepumi_clean_df = pd.read_csv(deepumi_clean, sep="\t", low_memory=False)

# Restrict to specified samples
dupcaller_df = dupcaller_df[dupcaller_df["SAMPLE_ID"].isin(dupcaller_samples)]
deepumi_df = deepumi_df[deepumi_df["SAMPLE_ID"].isin(samples)]
deepumi_clean_df = deepumi_clean_df[deepumi_clean_df["SAMPLE_ID"].isin(samples)]

# Add VARIANT_ID
dupcaller_df = add_variant_id(dupcaller_df)
deepumi_df = add_variant_id(deepumi_df)

# Mark PASS status for deepUMI using the clean somatic set
deepumi_df["is_pass"] = deepumi_df["VARIANT_ID"].isin(deepumi_clean_df["MUT_ID"])

# Harmonise mutation types
deepumi_df["TYPE"] = deepumi_df["TYPE"].replace({"INSERTION": "INDEL", "DELETION": "INDEL"})

# Optional: restrict to SNVs only
if mutations == "snv":
    deepumi_df = deepumi_df[deepumi_df["TYPE"] == "SNV"]
    dupcaller_df = dupcaller_df[dupcaller_df["TYPE"] == "SNV"]
elif mutations == "indel":
    deepumi_df = deepumi_df[deepumi_df["TYPE"] == "INDEL"]
    dupcaller_df = dupcaller_df[dupcaller_df["TYPE"] == "INDEL"]

print(f"deepUMI variants: {len(deepumi_df):,}  |  DupCaller variants: {len(dupcaller_df):,}")

deepUMI variants: 8,112  |  DupCaller variants: 10,150


In [6]:
# --- 4c. Identify DupCaller-unique mutations (PASS, F1R2>1, F2R1>1) ----------

merged_pass_both_df = build_merged_variants(
    deepumi_df, dupcaller_df, deepumi_pass_only=True, dupcaller_pass_only=True,
)
merged_all_df = build_merged_variants(
    deepumi_df, dupcaller_df, deepumi_pass_only=False, dupcaller_pass_only=False,
)

# Variants that PASS only in DupCaller
variants_pass_dupcaller_only = merged_pass_both_df[
    merged_pass_both_df["status"] == "Only DupCaller"
][["VARIANT_ID", "SAMPLE_ID"]]

variants_pass_deepumi = merged_pass_both_df[
    merged_pass_both_df["status"].isin(["Only deepUMI", "Common"])
][["VARIANT_ID", "SAMPLE_ID"]]

variants_all_deepumi = merged_all_df[
    merged_all_df["status"].isin(["Only deepUMI", "Common"])
][["VARIANT_ID", "SAMPLE_ID"]]

# Select variants that are not at all present in deepUMI
only_dupcaller_pass = variants_pass_dupcaller_only.merge(
    merged_all_df, on=["VARIANT_ID", "SAMPLE_ID"], how="inner"
)

exclude_idx = pd.MultiIndex.from_frame(variants_all_deepumi[["VARIANT_ID", "SAMPLE_ID"]])
keep_mask = ~pd.MultiIndex.from_frame(
    only_dupcaller_pass[["VARIANT_ID", "SAMPLE_ID"]]
).isin(exclude_idx)
only_dupcaller_pass = only_dupcaller_pass[keep_mask]

# Extract F1R2 and F2R1 from DupCaller INFO field
only_dupcaller_pass["F1R2"] = extract_info_field(
    only_dupcaller_pass["INFO_dupcaller"], "F1R2"
).astype("Int64")
only_dupcaller_pass["F2R1"] = extract_info_field(
    only_dupcaller_pass["INFO_dupcaller"], "F2R1"
).astype("Int64")

# Filter: both F1R2 > 1 and F2R1 > 1
filtered_only_dupcaller_pass = only_dupcaller_pass[
    (only_dupcaller_pass["F1R2"] > 1) & (only_dupcaller_pass["F2R1"] > 1)
].copy().reset_index(drop=True)

print(f"DupCaller-unique mutations (F1R2>1 & F2R1>1): {len(filtered_only_dupcaller_pass):,}")
for sample in samples:
    n = (filtered_only_dupcaller_pass["SAMPLE_ID"] == sample).sum()
    print(f"  {sample}: {n}")

DupCaller-unique mutations (F1R2>1 & F2R1>1): 1,460
  P19_0050_BDO_01: 289
  P19_0050_BTR_01: 293
  P19_0051_BDO_01: 183
  P19_0051_BTR_01: 272
  P19_0052_BDO_01: 62
  P19_0052_BTR_01: 70
  P19_0053_BDO_01: 161
  P19_0053_BTR_01: 130


## 5. Pileup functions

Two pileup functions for extracting per-position metrics from BAM files:

- **`pileup_position_basic`** — counts total, alt, ref, N, other. Used for the four deepUMIcaller pipeline BAMs.
- **`pileup_position_dupaware`** — same counts, plus split by duplicate flag (`dup_alt`, `nondup_alt`, etc.). Used for the DupCaller markdup BAM where GATK MarkDuplicates has set the `0x400` flag.

In [ ]:
# --- 5. Pileup functions ------------------------------------------------------
def pileup_position_basic(bam_or_path, chrom: str, pos: int, ref: str, alt: str) -> dict:
    """Pileup a single position — returns total, n_count, alt_count, ref_count, other_count.

    bam_or_path: either a file path (str) or an already-open pysam.AlignmentFile.
    Passing an open BAM avoids re-opening the index for every mutation."""
    total = n_count = alt_count = ref_count = other_count = 0
    _cm = (pysam.AlignmentFile(bam_or_path, "rb")
           if isinstance(bam_or_path, (str, os.PathLike))
           else contextlib.nullcontext(bam_or_path))
    with _cm as bam:
        for col in bam.pileup(chrom, pos - 1, pos, truncate=True,
                               min_base_quality=0, min_mapping_quality=0):
            if col.reference_pos != pos - 1:
                continue
            for pr in col.pileups:
                if pr.is_del or pr.is_refskip:
                    continue
                base = pr.alignment.query_sequence[pr.query_position]
                total += 1
                if base == "N":
                    n_count += 1
                elif base == alt:
                    alt_count += 1
                elif base == ref:
                    ref_count += 1
                else:
                    other_count += 1
    n_frac = n_count / total if total > 0 else float("nan")
    return {"total": total, "n_count": n_count, "n_frac": n_frac,
            "alt_count": alt_count, "ref_count": ref_count, "other_count": other_count}


def pileup_position_dupaware(bam_or_path, chrom: str, pos: int, ref: str, alt: str) -> dict:
    """Pileup a single position, splitting counts by the GATK duplicate flag (0x400).

    Returns the same columns as pileup_position_basic plus:
      dup_alt, dup_ref       — counts from reads with is_duplicate=True
      nondup_alt, nondup_ref — counts from reads with is_duplicate=False

    This distinguishes ALT support that GATK kept as primary from ALT support
    that is only in duplicate-flagged reads (which DupCaller includes in
    genotyping but extractDepthSnv skips).

    bam_or_path: either a file path (str) or an already-open pysam.AlignmentFile."""
    total = n_count = alt_count = ref_count = other_count = 0
    dup_alt = dup_ref = nondup_alt = nondup_ref = 0

    _cm = (pysam.AlignmentFile(bam_or_path, "rb")
           if isinstance(bam_or_path, (str, os.PathLike))
           else contextlib.nullcontext(bam_or_path))
    with _cm as bam:
        for col in bam.pileup(chrom, pos - 1, pos, truncate=True,
                               min_base_quality=0, min_mapping_quality=0):
            if col.reference_pos != pos - 1:
                continue
            for pr in col.pileups:
                if pr.is_del or pr.is_refskip:
                    continue
                read = pr.alignment
                if read.is_secondary or read.is_supplementary:
                    continue
                base = read.query_sequence[pr.query_position]
                is_dup = read.is_duplicate
                total += 1

                if base == "N":
                    n_count += 1
                elif base == alt:
                    alt_count += 1
                    if is_dup:
                        dup_alt += 1
                    else:
                        nondup_alt += 1
                elif base == ref:
                    ref_count += 1
                    if is_dup:
                        dup_ref += 1
                    else:
                        nondup_ref += 1
                else:
                    other_count += 1

    n_frac = n_count / total if total > 0 else float("nan")
    return {
        "total": total, "n_count": n_count, "n_frac": n_frac,
        "alt_count": alt_count, "ref_count": ref_count, "other_count": other_count,
        "dup_alt": dup_alt, "nondup_alt": nondup_alt,
        "dup_ref": dup_ref, "nondup_ref": nondup_ref,
    }

## 6. Generic parallel pileup runner

A reusable function that pileups all DupCaller-unique mutations against any BAM,
using `ThreadPoolExecutor` for speed (one BAM handle per sample).
All subsequent pileup sections call this function.

In [8]:
# --- 6. Generic parallel pileup runner ----------------------------------------

N_WORKERS = min(len(samples), 8)  # one thread per sample; request ≥8 CPUs


def _run_pileup_for_sample(sample, bam_path_fn, pileup_fn, out_dir, mutation_df):
    """Pileup all mutations for one sample. Opens BAM once for efficiency.

    Parameters
    ----------
    sample : str
    bam_path_fn : callable  – sample → BAM path
    pileup_fn : callable    – (bam_handle, chrom, pos, ref, alt) → dict
    out_dir : Path
    mutation_df : pd.DataFrame with VARIANT_ID column
    """
    bam_path = bam_path_fn(sample)
    if not os.path.exists(bam_path):
        return sample, None, f"BAM not found: {bam_path}"

    records = []
    with pysam.AlignmentFile(bam_path, "rb") as bam:
        for _, row in mutation_df.iterrows():
            chrom, pos, ref, alt = parse_variant_id(row["VARIANT_ID"])
            result = pileup_fn(bam, chrom, pos, ref, alt)
            records.append({"VARIANT_ID": row["VARIANT_ID"], **result})

    df = pd.DataFrame(records)
    df.to_csv(out_dir / f"{sample}.tsv", sep="\t", index=False)
    return sample, df, "ok"


def run_parallel_pileup(label, out_folder, bam_path_fn, pileup_fn, mutation_source):
    """Run pileup in parallel across all samples.

    Parameters
    ----------
    label : str               – human-readable name for logging
    out_folder : str           – subfolder under OUTPUT_BASE
    bam_path_fn : callable     – sample → BAM path
    pileup_fn : callable       – (bam_handle, chrom, pos, ref, alt) → dict
    mutation_source : dict     – {sample: DataFrame} with the mutations to pileup

    Returns
    -------
    dict[str, pd.DataFrame]
    """
    out_dir = OUTPUT_BASE / out_folder
    out_dir.mkdir(parents=True, exist_ok=True)

    results = {}
    print(f"\n{'═' * 80}")
    print(f"Pileup: {label}  →  {out_folder}/")
    print(f"{'─' * 80}")

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {}
        for sample in samples:
            if sample not in mutation_source:
                continue
            muts = mutation_source[sample]
            futures[executor.submit(
                _run_pileup_for_sample, sample, bam_path_fn, pileup_fn, out_dir, muts
            )] = sample

        for future in as_completed(futures):
            sample, df, status = future.result()
            if status == "ok":
                results[sample] = df
                n = len(df)
                alt0 = (df["alt_count"] == 0).sum()
                print(f"  [{sample}] {n} muts | ALT=0: {alt0} ({alt0/n*100:.0f}%) | "
                      f"ALT>0: {n - alt0} ({(n - alt0)/n*100:.0f}%)")
            else:
                print(f"  [{sample}] Skipped — {status}")

    print(f"Saved to: {out_dir}/")
    return results

In [9]:
# --- Build per-sample mutation DataFrames (used as input for all pileups) ------
mutation_source = {}
for sample in samples:
    sample_muts = filtered_only_dupcaller_pass[
        filtered_only_dupcaller_pass["SAMPLE_ID"] == sample
    ].copy()
    if not sample_muts.empty:
        mutation_source[sample] = sample_muts
        print(f"[{sample}] {len(sample_muts)} DupCaller-unique mutations")
    else:
        print(f"[{sample}] No DupCaller-unique mutations — will be skipped")

[P19_0050_BDO_01] 289 DupCaller-unique mutations
[P19_0050_BTR_01] 293 DupCaller-unique mutations
[P19_0051_BDO_01] 183 DupCaller-unique mutations
[P19_0051_BTR_01] 272 DupCaller-unique mutations
[P19_0052_BDO_01] 62 DupCaller-unique mutations
[P19_0052_BTR_01] 70 DupCaller-unique mutations
[P19_0053_BDO_01] 161 DupCaller-unique mutations
[P19_0053_BTR_01] 130 DupCaller-unique mutations


## 7. Extract pileups from all BAMs

Run `pileup_position_basic` for each DupCaller-unique mutation against the four
deepUMIcaller pipeline BAMs, and `pileup_position_dupaware` for the DupCaller markdup BAM.
Each call uses `ThreadPoolExecutor` for parallelism.

In [ ]:
# --- 7a. Pileup: raw BAM (sortbamclean) ──────────────────────────────────────
raw_bam_pileups = run_parallel_pileup(
    label="Raw aligned reads (sortbamclean)",
    out_folder="0_raw_bam",
    bam_path_fn=get_raw_bam,
    pileup_fn=pileup_position_basic,
    mutation_source=mutation_source,
)

### 7b. Filtered consensus BAM (`sortbamduplexfiltered`)

Consensus reads after AS-XS filtering but **before** `FilterConsensusReads` and `ClipBam`.
This is the last BAM in the pipeline without clipping — comparing with the next two BAMs
isolates the effect of FilterConsensusReads + ClipBam.

In [ ]:
# --- 7b. Pileup: filtered consensus BAM (sortbamduplexfiltered) ───────────────
filtered_consensus_pileups = run_parallel_pileup(
    label="Filtered consensus (sortbamduplexfiltered)",
    out_folder="1_sortbamduplexfiltered",
    bam_path_fn=get_filtered_consensus_bam,
    pileup_fn=pileup_position_basic,
    mutation_source=mutation_source,
)

### 7c. Sort BAM duplex clean (`sortbamduplexclean`)

Lenient AM consensus (≥2 reads, single-strand OK), **after** `FilterConsensusReads`.
Comparing `sortbamduplexfiltered` → `sortbamduplexclean`.

In [ ]:
# --- 7c. Pileup: all-molecules consensus BAM (sortbamduplexclean) ─────────────
all_molecules_pileups = run_parallel_pileup(
    label="All-molecules consensus (sortbamduplexclean)",
    out_folder="2_sortbamduplexclean",
    bam_path_fn=get_all_molecules_bam,
    pileup_fn=pileup_position_basic,
    mutation_source=mutation_source,
)

### 7d. Duplex consensus BAM (`sortbamduplexconsmed`)

Strict duplex consensus (≥4 reads, both strands required), **after** `FilterConsensusReadsDuplex` + `ClipBam`.
This is the final BAM used by deepUMIcaller for variant calling with VarDict.

In [ ]:
# --- 7d. Pileup: duplex consensus BAM (sortbamduplexconsmed) ──────────────────
duplex_consensus_pileups = run_parallel_pileup(
    label="Duplex consensus (sortbamduplexconsmed)",
    out_folder="3_sortbamduplexconsmed",
    bam_path_fn=get_duplex_consensus_bam,
    pileup_fn=pileup_position_basic,
    mutation_source=mutation_source,
)

### 7e. DupCaller markdup BAM (duplicate-aware)

Raw markdup BAM from DupCaller. Uses `pileup_position_dupaware` to split read counts
by the GATK `0x400` duplicate flag, so we can see whether ALT support comes from
primary or duplicate-flagged reads.

In [10]:
# --- 7e. Pileup: DupCaller markdup BAM (duplicate-aware) ──────────────────────
dupcaller_markdup_pileups = run_parallel_pileup(
    label="DupCaller markdup BAM (dupaware)",
    out_folder="dupcaller_markdup_bam",
    bam_path_fn=get_dupcaller_markdup_bam,
    pileup_fn=pileup_position_dupaware,
    mutation_source=mutation_source,
)


════════════════════════════════════════════════════════════════════════════════
Pileup: DupCaller markdup BAM (dupaware)  →  dupcaller_markdup_bam/
────────────────────────────────────────────────────────────────────────────────


[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0053_BTR_01/markdup/P19_0053_BTR_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0050_BTR_01/markdup/P19_0050_BTR_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0050_BDO_01/markdup/P19_0050_BDO_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0053_BDO_01/markdup/P19_0053_BDO_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0051_BTR_01/markdup/P19_0051_BTR_01.mkdped.bam.bai
[W::hts_id

  [P19_0052_BTR_01] 70 muts | ALT=0: 1 (1%) | ALT>0: 69 (99%)


[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0052_BDO_01/markdup/P19_0052_BDO_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0053_BDO_01/markdup/P19_0053_BDO_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0053_BTR_01/markdup/P19_0053_BTR_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0052_BDO_01/markdup/P19_0052_BDO_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0052_BDO_01/markdup/P19_0052_BDO_01.mkdped.bam.bai
[W::hts_id

  [P19_0052_BDO_01] 62 muts | ALT=0: 4 (6%) | ALT>0: 58 (94%)


...

  [P19_0050_BDO_01] 289 muts | ALT=0: 28 (10%) | ALT>0: 261 (90%)


[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0050_BTR_01/markdup/P19_0050_BTR_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0050_BTR_01/markdup/P19_0050_BTR_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0051_BTR_01/markdup/P19_0051_BTR_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0050_BTR_01/markdup/P19_0050_BTR_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0050_BTR_01/markdup/P19_0050_BTR_01.mkdped.bam.bai
[W::hts_id

  [P19_0051_BTR_01] 272 muts | ALT=0: 24 (9%) | ALT>0: 248 (91%)


[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0050_BTR_01/markdup/P19_0050_BTR_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0050_BTR_01/markdup/P19_0050_BTR_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0050_BTR_01/markdup/P19_0050_BTR_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0050_BTR_01/markdup/P19_0050_BTR_01.mkdped.bam.bai
[W::hts_idx_load3] The index file is older than the data file: /data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/P19_0050_BTR_01/markdup/P19_0050_BTR_01.mkdped.bam.bai
[W::hts_id

  [P19_0050_BTR_01] 293 muts | ALT=0: 23 (8%) | ALT>0: 270 (92%)
Saved to: results/dupcaller_unique/dupcaller_markdup_bam/


## 8. Reload from TSV (optional)

If the pileups were already generated, reload them from TSV files without BAM access.

In [11]:
# --- 8. Reload all pileup TSVs from disk (optional) ──────────────────────────
def _reload_tsvs(subfolder: str, label: str) -> dict[str, pd.DataFrame]:
    """Load per-sample TSVs from OUTPUT_BASE / subfolder."""
    d = OUTPUT_BASE / subfolder
    result = {}
    for sample in samples:
        path = d / f"{sample}.tsv"
        if path.exists():
            result[sample] = pd.read_csv(path, sep="\t")
            print(f"  [{sample}] {label}: {len(result[sample])} rows")
        else:
            print(f"  [{sample}] {label}: not found")
    return result

print("Reloading all TSVs from disk …\n")

raw_bam_pileups            = _reload_tsvs("0_raw_bam",                "Raw BAM")
print()
filtered_consensus_pileups = _reload_tsvs("1_sortbamduplexfiltered",  "Filtered consensus BAM")
print()
all_molecules_pileups      = _reload_tsvs("2_sortbamduplexclean",     "All-molecules BAM")
print()
duplex_consensus_pileups   = _reload_tsvs("3_sortbamduplexconsmed",   "Duplex consensus BAM")
print()
dupcaller_markdup_pileups  = _reload_tsvs("dupcaller_markdup_bam",    "DupCaller markdup BAM (dupaware)")
print()

print("\n✅ All TSVs reloaded.")

Reloading all TSVs from disk …

  [P19_0050_BDO_01] Raw BAM: 289 rows
  [P19_0050_BTR_01] Raw BAM: 293 rows
  [P19_0051_BDO_01] Raw BAM: 183 rows
  [P19_0051_BTR_01] Raw BAM: 272 rows
  [P19_0052_BDO_01] Raw BAM: 62 rows
  [P19_0052_BTR_01] Raw BAM: 70 rows
  [P19_0053_BDO_01] Raw BAM: 161 rows
  [P19_0053_BTR_01] Raw BAM: 130 rows

  [P19_0050_BDO_01] Filtered consensus BAM: 289 rows
  [P19_0050_BTR_01] Filtered consensus BAM: 293 rows
  [P19_0051_BDO_01] Filtered consensus BAM: 183 rows
  [P19_0051_BTR_01] Filtered consensus BAM: 272 rows
  [P19_0052_BDO_01] Filtered consensus BAM: 62 rows
  [P19_0052_BTR_01] Filtered consensus BAM: 70 rows
  [P19_0053_BDO_01] Filtered consensus BAM: 161 rows
  [P19_0053_BTR_01] Filtered consensus BAM: 130 rows

  [P19_0050_BDO_01] All-molecules BAM: 289 rows
  [P19_0050_BTR_01] All-molecules BAM: 293 rows
  [P19_0051_BDO_01] All-molecules BAM: 183 rows
  [P19_0051_BTR_01] All-molecules BAM: 272 rows
  [P19_0052_BDO_01] All-molecules BAM: 62 rows
  [

## Summary

All TSV files are saved under `results/dupcaller_unique/`:

In [12]:
# --- Final check: list all generated files ------------------------------------
for subdir in ["0_raw_bam", "1_sortbamduplexfiltered", "2_sortbamduplexclean",
               "3_sortbamduplexconsmed", "dupcaller_markdup_bam"]:
    d = OUTPUT_BASE / subdir
    if d.exists():
        files = sorted(d.glob("*.tsv"))
        print(f"\n{subdir}/ ({len(files)} files):")
        for f in files:
            print(f"  {f.name}")
    else:
        print(f"\n{subdir}/ — NOT CREATED")


0_raw_bam/ (8 files):
  P19_0050_BDO_01.tsv
  P19_0050_BTR_01.tsv
  P19_0051_BDO_01.tsv
  P19_0051_BTR_01.tsv
  P19_0052_BDO_01.tsv
  P19_0052_BTR_01.tsv
  P19_0053_BDO_01.tsv
  P19_0053_BTR_01.tsv

1_sortbamduplexfiltered/ (8 files):
  P19_0050_BDO_01.tsv
  P19_0050_BTR_01.tsv
  P19_0051_BDO_01.tsv
  P19_0051_BTR_01.tsv
  P19_0052_BDO_01.tsv
  P19_0052_BTR_01.tsv
  P19_0053_BDO_01.tsv
  P19_0053_BTR_01.tsv

2_sortbamduplexclean/ (8 files):
  P19_0050_BDO_01.tsv
  P19_0050_BTR_01.tsv
  P19_0051_BDO_01.tsv
  P19_0051_BTR_01.tsv
  P19_0052_BDO_01.tsv
  P19_0052_BTR_01.tsv
  P19_0053_BDO_01.tsv
  P19_0053_BTR_01.tsv

3_sortbamduplexconsmed/ (8 files):
  P19_0050_BDO_01.tsv
  P19_0050_BTR_01.tsv
  P19_0051_BDO_01.tsv
  P19_0051_BTR_01.tsv
  P19_0052_BDO_01.tsv
  P19_0052_BTR_01.tsv
  P19_0053_BDO_01.tsv
  P19_0053_BTR_01.tsv

dupcaller_markdup_bam/ (8 files):
  P19_0050_BDO_01.tsv
  P19_0050_BTR_01.tsv
  P19_0051_BDO_01.tsv
  P19_0051_BTR_01.tsv
  P19_0052_BDO_01.tsv
  P19_0052_BTR_01.tsv